# INSTALL LIBRARY UNSLOTH

In [1]:
%%capture
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install trl bitsandbytes
!pip install --upgrade transformers
!pip install --upgrade torchao

# IMPORT LOGIN USING TOKEN

In [ ]:
from huggingface_hub import login

login(token="")

# MODEL SELECTION

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "opsirygi/Azhale0.1", # Change this to unsloth/gemma-4-E2B-it etc
    dtype = None, # None for auto detection
    max_seq_length = 8000, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

# LOAD DATASET FOR TRAINING AND CHAT TEMPLATE

In [ ]:
from datasets import load_dataset, Dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats

dataset = load_dataset("", split="train")
                       
dataset = standardize_data_formats(dataset)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking", 
)

def format_prompt(ex):
    return {"text": f"<|turn>system\n<turn|>\n<|turn>user\n{ex['messages'][0]['content']}<turn|>\n<|turn>model\n{ex['messages'][1]['content']}<turn|>"}

dataset.save_to_disk("")
dataset = dataset.map(format_prompt)

In [ ]:
print(dataset.column_names)

print(dataset[0]["text"][:120]) 

In [ ]:
model = FastModel.get_peft_model (
    model,
    finetune_vision_layer=False,
    finetune_language_layer=True,
    finetune_attention_modules=True,

    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout= 0,
    use_gradient_checkpointing="unsloth",
    use_rslora=False,
    loftq_config=None,
    bias="none",
    random_state=3407
)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

Dataset = Dataset.load_from_disk("")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=None,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        num_train_epochs=5,
        max_steps=-1,
        learning_rate=2e-4,
        logging_steps=100,
        optim= "adamw_8bit",
        dataset_text_field="text",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none"
    )
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n"
)

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

# TRAINING MODEL

In [ ]:
trainer.train()

# EVAL MODEL

In [ ]:
FastModel.for_inference(model)

messages = [{"role": "system",
    "role": "user",
    "content": "pagi, apa kamu ada reasoning?, oh iya?"
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
).to("cuda")

outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=128,
    use_cache=True,
    do_sample=True,
    temperature=0.7,
    top_k=64,
    top_p=0.95
)

response = tokenizer.batch_decode(outputs)[0]
print(response)

# SAVE MODEL TO LOCAL

In [ ]:
model.save_pretrained("")
tokenizer.save_pretrained("")

# SAVE MODEL TO HUGGING FACE

In [ ]:
model.push_to_hub_merged("", tokenizer, save_method="merged_16bit")
tokenizer.push_to_hub("")